In [1]:
import neuroglancer
import pandas as pd
import numpy as np
from tqdm import tqdm
import re
import caveclient
from nglui import statebuilder
from nglui.statebuilder import *
import navis
import cloudvolume as cv
import ast
import json
import pymaid

In [2]:
client = caveclient.CAVEclient(server_address='https://global.connectomics.braininbrain.org')

# catmaid auth
path = 'D:/flywire_backup/cave/cat_client.json' # replace with your path

with open(path, 'r') as f:
    conn_file = json.load(f)
api_token = conn_file['api_token']
http_user = conn_file['http_user']
http_password = conn_file['http_password']
server = conn_file['server']

cat_client = pymaid.CatmaidInstance(server = server,  # update with your auth
                                api_token = api_token,
                                caching = True,
                                project_id = 8,
                                http_user = http_user,
                                http_password = http_password 
                                )

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)
INFO:pymaid:Global CATMAID instance set. Caching is ON.


## data import

In [4]:
species = 'megalopta' # megalopta or eciton
datastack = 'EB' 
roi = 'EB'

### get synapses (if plotting otherwise ignore section)

In [5]:
if species == 'megalopta':
    syntable = pd.read_csv('../syntables/table_csv_files/megalopta_cx_syntable_19-04-26.csv')
    #syntable = pd.read_csv('../syntables/table_csv_files/megalopta_cx_syntable_06-12-25.csv')
    
    for c in tqdm([c for c in syntable.columns if 'coords' in c or 'vox' in c]): 
        syntable[c] = syntable[c].apply(lambda s: tuple(map(int, s.strip('()').split(','))))
    
    
    # syntable = syntable.rename(columns={
    #     'pre_vox_coord': 'pre_coord',
    #     'post_vox_coord': 'post_coord',
    #     'pre_neuron_name': 'pre_name',
    #     'post_neuron_name': 'post_name'
    # })

elif species == 'eciton':
    # syntable = pd.read_csv('../syntables/table_csv_files/eciton_cx_syntable_27-12-25.csv') 
    syntable = pd.read_csv('../syntables/table_csv_files/eciton_delta7.csv') 
    for c in tqdm([c for c in syntable.columns if 'coords' in c or 'vox' in c]): 
        syntable[c] = syntable[c].apply(lambda s: tuple(map(int, s.strip('()').split(','))))
    
    syntable = syntable[syntable['pre_skid'] != syntable['post_skid']] #remove autapses
    
    syntable = syntable.rename(columns={
        'pre_vox_coord': 'pre_coord',
        'post_vox_coord': 'post_coord',
        'pre_neuron_name': 'pre_name',
        'post_neuron_name': 'post_name'
    })


C:\Users\Marcel\AppData\Local\Temp\ipykernel_10944\2913174682.py:2: DtypeWarning: Columns (2,3,9,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  syntable = pd.read_csv('../syntables/table_csv_files/megalopta_cx_syntable_19-04-26.csv')
0it [00:00, ?it/s]


### format coords (can skip if syntable is nglformat version)

In [6]:
'''run if coords arent tuples'''

def fast_parse_coord_col(s: "pd.Series"):
    out = s.copy()

    mask = out.map(type).eq(str)  # only strings
    if not mask.any():
        return out

    extracted = out[mask].str.extract(
        r'(-?\d+(?:\.\d+)?)\s*[,\s]\s*(-?\d+(?:\.\d+)?)\s*[,\s]\s*(-?\d+(?:\.\d+)?)'
    )

    # to numeric arrays
    arr = extracted.astype(float).to_numpy()

    # build tuples (still Python objects, but only for the subset)
    out.loc[mask] = list(map(tuple, arr))
    return out

syntable["pre_coord"]  = fast_parse_coord_col(syntable["pre_coord"])
syntable["post_coord"] = fast_parse_coord_col(syntable["post_coord"])


bad = syntable[syntable["pre_coord"].isna() | syntable["post_coord"].isna()]
print(len(bad), "rows have missing coords after parsing")

for c in ["pre_skid", "post_skid"]:
    syntable[c] = (
        pd.to_numeric(syntable[c], errors="coerce")  # handles "54800.0", "5.48e4", etc.
        .round()                                     # just in case
        .astype("Int64")                             # keeps NA as <NA>
    )

0 rows have missing coords after parsing


In [ ]:
# for checking electronic analysis synapses

# syntable = pd.read_csv('epg_l2_connectors_electrotonicDist.csv')

# # Merge columns x, y, z into a single column 'post_coord' with format (x, y, z)

# syntable['x'] = syntable['x'] / 10
# syntable['y'] = syntable['y'] / 10
# syntable['z'] = syntable['z'] / 50

# syntable['post_coord'] = syntable.apply(lambda row: f"({row['x']}, {row['y']}, {row['z']})", axis=1)
# syntable['post_coord'] = syntable.apply(
#     lambda row: (int(row['x']), int(row['y']), int(row['z'])), axis=1
# )
# syntable.head()

### get neuron nametable with CAVE root ids

In [19]:
datastack = 'PB1'

In [20]:
if species == "megalopta":
    if datastack == 'FBEB':
        epg = pd.read_csv('../syntables/updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_EPG.csv', dtype={'Root ID': str})
        pen = pd.read_csv('../syntables/updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_PEN.csv', dtype={'Root ID': str})
        er = pd.read_csv('../syntables/updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_ER.csv', dtype={'Root ID': str})
        #er = er[er['Completed']==True]
        nametable = pd.concat((epg, pen, er))
    elif datastack == 'PB1':
        epg = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_EPG.csv', dtype={'Root ID': str})
        pen = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_PEN.csv', dtype={'Root ID': str}) 
        d7 = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_delta7.csv', dtype={'Root ID': str})
        nametable = pd.concat((epg, pen, d7))
    elif datastack == 'PB2':
        d7 = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB2_preprint_FINAL - PB2_delta7.csv', dtype={'Root ID': str})
        epg = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB2_preprint_FINAL - PB2_EPG.csv', dtype={'Root ID': str})
        nametable = pd.concat((d7, epg))
    elif datastack == 'NO':
        lno = pd.read_csv('../syntables/updated_google_sheets/Megalopta_NOr neuron_CAVE progress - NOr_LNO.csv', dtype={'Root ID': str})
        pen = pd.read_csv('../syntables/updated_google_sheets/Megalopta_NOr neuron_CAVE progress - NOr_PEN.csv', dtype={'Root ID': str})
        nametable = pd.concat((lno, pen))
    
elif species == "eciton":
    d7 = pd.read_csv('../syntables/updated_google_sheets/Eciton_PB neuron_CAVE progress - eciton_PB_Delta7.csv', dtype={'Root ID': str})
    epg = pd.read_csv('../syntables/updated_google_sheets/Eciton_PB neuron_CAVE progress - eciton_PB_EPG.csv', dtype={'Root ID': str})
    nametable_pb = pd.concat((epg, d7))
    nametable_pb["roi"] = "PB"

    nametable = nametable_pb.copy()

nametable = nametable[['Catmaid name', 'Root ID', 'CATMAID skid']].rename(columns={'Catmaid name':'name', 'Root ID':'root_ids', 'CATMAID skid':'skid'})

nametable['skid'] = nametable['skid'].fillna(0).astype(float).astype(int)

# make sure to fill na
nametable['root_ids'] = nametable['root_ids'].fillna('')

# convert each cell in 'root_ids' from a comma-separated string to a list of integers, ignoring empty strings
nametable['root_ids'] = nametable['root_ids'].apply(lambda x: [int(seg) for seg in x.split(',') if seg.strip()])

# remove empty values
nametable = nametable[nametable['root_ids'].apply(lambda x: len(x) > 0)].copy()

'''NOTE this removes any neuron with skid == 0. This is to remove unproofread cells...if you 
want to keep these neurons, you have to add a skid otherwise the update will not work properly.'''
nametable = nametable[nametable['skid'] != 0]

nametable["skid"] = nametable["skid"].astype("Int64")

nametable

,name,root_ids,skid
0,EPG_L2_55944,[576460752521344750],55944
1,EPG_L2_55952,[576460752510496133],55952
2,EPG_L3_54939,"[576460752494001689, 576460752504703640]",54939
3,EPG_L3_57518,"[576460752372935125, 576460752440175329, 57646...",57518
4,EPG_L3_57529,"[576460752460921048, 576460752509771516]",57529
...,...,...,...
37,delta7_R_L2L10R7_83213,[576460752550908713],83213
38,delta7_R_L2L10R7_83722,[576460752608580253],83722
39,delta7_R_L2L10R7_84178,[576460752490172734],84178
40,delta7_R_L2L10R7_84473,"[576460752543874747, 576460752545051043]",84473


## statebuilder (generates neuroglancer states)

In [30]:
# get skids of neurons to vis
nametable[nametable['name'].str.contains(r'PEN')]

,name,root_ids,skid
0,PEN_a_L2_112713,[576460752489132854],112713
1,PEN_b_L2_112687,[576460752507193318],112687
2,PEN_b_L3_63772,[576460752591088521],63772
3,PEN_a_L3_63776,[576460752594006960],63776
4,PEN_a_L4_35368,[576460752596578281],35368
5,PEN_b_L4_99084,[576460752478055195],99084
6,PEN_a_L4_35091,[576460752535931016],35091
7,PEN_b_L4_35224,[576460752508292053],35224
8,PEN_b_L5_35102,"[576460752523851287, 576460752560317598, 57646...",35102
9,PEN_a_L5_34931,[576460752516169926],34931


In [32]:
'''Synapse plotter'''

'''USER INPUT'''
skid_list = [63772] #84617
roi = r"PB"
view = "normal" # can ignore, just testing...synapses don't work for spleunker for some reason

#######################
if species == "megalopta":
    if roi == 'EB':
        seg_source='graphene://middleauth+https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_EB_v3/'
        print('graphene set for megalopta FB / EB data')
    elif roi == 'PB':
        if view == "normal":
            seg_source='graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a'
            print('graphene set for megalopta PB data')
        elif view == "spleunker":
            seg_source='graphene://middleauth+https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a'
    elif roi == 'PB2':
        seg_source='graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB2'
elif species == "eciton":
    if roi == 'PB':
        seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_PB_v1'
        print('graphene set for eciton PB data')
    elif roi == 'FB':
        seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_FB'
        print('graphene set for eciton FB / EB data')
    elif roi == 'NO':
        seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/eciton_NO_v5'
        print('graphene set for eciton Noduli data')

n = nametable[nametable['skid'].isin(skid_list)]
root_list = (n['root_ids'].explode().tolist())

def to_int_scalar(x):
    # unwrap list/tuple/np.array of length 1
    if isinstance(x, (list, tuple, np.ndarray)):
        if len(x) == 0:
            return None
        x = x[0]
    # unwrap pandas/numpy scalar
    if x is None:
        return None
    return int(x)

root_list = [to_int_scalar(r) for r in root_list]
root_list = [r for r in root_list if r is not None]
print("root_list:", root_list, [type(r) for r in root_list])

# # count the number of segments
count = len(root_list)  
print(f'There are {count} segments')


# Set up viewer options
view_options = {
    "background_color": "white",
}
    # "position": [
    #     24090.462890625,
    #     22922.3984375,
    #     3824.824951171875
    # ]


# Image layer configuration
if roi == "PB2":
    img_src = 'precomputed://https://lweb1569.srv.lu.se/catmaid/data/megalopta/Megalopta_PB_PT2_256'
elif roi == "PB":
    img_src = "precomputed://https://lweb1569.srv.lu.se/catmaid/data/megalopta/Megalopta_PB_PT1_512"
elif roi == "EB":
    img_src = "precomputed://https://lweb1569.srv.lu.se/catmaid/data/megalopta/Megalopta_FB-EB_512"
else: 
    img_src = 'precomputed://http://localhost:8000'
#lund_img_source = 'precomputed://https://lweb1569.srv.lu.se/catmaid/data/eciton/Eciton_PB_1026'
img_layer = ImageLayerConfig(name='layer23', source=img_src)


# Segmentation layer configuration with fixed IDs and color mapping
# 'graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_FB'

seg_layer = SegmentationLayerConfig(
    name='seg',
    source=seg_source,
    fixed_ids=root_list  # Use flattened segment IDs - doesn't use df below (in renderstate)
    # fixed_id_colors=color_list  # Color is based on 'color' column from syntable
)


syntable_filt = syntable[syntable["roi"].str.contains(roi, na=False)]

syntable_filt = syntable_filt[
    syntable_filt["pre_skid"].isin(skid_list).fillna(False) |
    syntable_filt["post_skid"].isin(skid_list).fillna(False)
]

pre = syntable_filt[syntable_filt["pre_skid"].isin(skid_list).fillna(False)].copy()
post = syntable_filt[syntable_filt["post_skid"].isin(skid_list).fillna(False)].copy()

pre["post_coord"] = [(0, 0, 0)] * len(pre)
post["pre_coord"] = [(0, 0, 0)] * len(post)

syntable_filt = pd.concat([pre, post], ignore_index=True)


lines = LineMapper(point_column_a="pre_coord", # use this if you want points connected by lines
                   point_column_b="post_coord")

pre_point = PointMapper(point_column='pre_coord') #to viz pre/post separately, use two anno tabs with point ann
post_point= PointMapper(point_column='post_coord')

anno_layer_lines = AnnotationLayerConfig(
    name='pre',
    color='red',
    mapping_rules=lines) 
anno_layer_pre = AnnotationLayerConfig(
    name='pre',
    color='red',
    mapping_rules=pre_point) 
anno_layer_post = AnnotationLayerConfig(
    name='post',
    color='cyan',
    mapping_rules=post_point)


# Build the state with layers and view options
sb = StateBuilder(layers=[img_layer,seg_layer,anno_layer_pre,anno_layer_post], view_kws=view_options) #seg_layer, img_layer anno_layer

# Render the state as HTML
sb.render_state(syntable_filt, return_as='html', link_text='State')



graphene set for megalopta PB data
root_list: [576460752591088521] [<class 'int'>]
There are 1 segments


In [93]:
# compare 
syntable.to_csv('../syntables/table_csv_files/megalopta_syntable_19-04-26_nglformat.csv')

In [30]:
syntable['roi'].unique()

skid_list = [54939]

array(['EB', 'PB', 'NO'], dtype=object)

In [40]:
# get skids of neurons to vis
skids = nametable[nametable['name'].str.contains(r'L3R6')]
skid_list = skids['skid'].to_list()
skids

,name,root_ids,skid
31,delta7_R_L3R6_82590,[576460752507668205],82590
32,delta7_R_L3R6_84354,[576460752456808411],84354
33,delta7_R_L3R6_84631,[576460752550433228],84631
34,delta7_R_L3R6_79852,[576460752524489121],79852
35,delta7_R_L3R6_81527,[576460752591033681],81527
36,delta7_R_L3R6_81541,"[576460752488914742, 576460752532095661]",81541


In [102]:
skid_list = [84631]

In [94]:
new_syntable = syntable.copy()
syntable = old_syntable.copy()

In [103]:
print(f'old syntable: {len(syntable[syntable["pre_skid"].isin(skid_list).fillna(False)])}')
print(f'new syntable: {len(new_syntable[new_syntable["pre_skid"].isin(skid_list).fillna(False)])}')

old syntable: 10081
new syntable: 10033


In [104]:
print(f'old syntable: {len(syntable[syntable["post_skid"].isin(skid_list).fillna(False)])}')
print(f'new syntable: {len(new_syntable[new_syntable["post_skid"].isin(skid_list).fillna(False)])}')

old syntable: 620
new syntable: 616


In [98]:
syntable = syntable[syntable['roi'].str.contains(r'PB', na=False)]
new_syntable = new_syntable[new_syntable['roi'].str.contains(r'PB', na=False)]

In [99]:
print(type(syntable["pre_coord"].iloc[0]), syntable["pre_coord"].iloc[0])
print(type(new_syntable["pre_coord"].iloc[0]), new_syntable["pre_coord"].iloc[0])

print(type(syntable["post_coord"].iloc[0]), syntable["post_coord"].iloc[0])
print(type(new_syntable["post_coord"].iloc[0]), new_syntable["post_coord"].iloc[0])

<class 'tuple'> (32140.0, 6587.0, 1723.0)
<class 'tuple'> (29539.0, 4469.0, 1872.0)
<class 'tuple'> (32163.0, 6578.0, 1721.0)
<class 'tuple'> (29526.0, 4495.0, 1872.0)


In [114]:
has_dups = new_syntable.duplicated(subset=["pre_coord", "post_coord"]).any()
print(has_dups)

False


In [111]:
len(new_syntable)

671066

In [112]:
len(old_syntable)

301569

In [108]:
old_df = syntable.copy()
new_df = new_syntable.copy()

old_df["syn_key"] = list(zip(old_df["pre_coord"], old_df["post_coord"]))
new_df["syn_key"] = list(zip(new_df["pre_coord"], new_df["post_coord"]))

missing_in_new = old_df[~old_df["syn_key"].isin(set(new_df["syn_key"]))].copy()
missing_in_old = new_df[~new_df["syn_key"].isin(set(old_df["syn_key"]))].copy()

old_keys = set(old_df["syn_key"])
new_keys = set(new_df["syn_key"])

missing_in_new = old_df[~old_df["syn_key"].isin(new_keys)].copy()
missing_in_old = new_df[~new_df["syn_key"].isin(old_keys)].copy()

print(f"missing synapses in new: {len(missing_in_new)}")
print(f"missing synapses in old: {len(missing_in_old)}")

missing synapses in new: 1271
missing synapses in old: 370768


In [116]:
print("old total rows:", len(old_df))
print("new total rows:", len(new_df))

print("old unique syn_key:", old_df["syn_key"].nunique())
print("new unique syn_key:", new_df["syn_key"].nunique())

old total rows: 301569
new total rows: 671066
old unique syn_key: 301569
new unique syn_key: 671066


In [117]:
new_dup_counts = new_df["syn_key"].value_counts()
print("new duplicated syn_keys:", (new_dup_counts > 1).sum())
print("max duplicate count in new:", new_dup_counts.max())

old_dup_counts = old_df["syn_key"].value_counts()
print("old duplicated syn_keys:", (old_dup_counts > 1).sum())
print("max duplicate count in old:", old_dup_counts.max())

new duplicated syn_keys: 0
max duplicate count in new: 1
old duplicated syn_keys: 0
max duplicate count in old: 1


In [118]:
dup_keys_new = new_dup_counts[new_dup_counts > 1].head(10).index
new_df[new_df["syn_key"].isin(dup_keys_new)].sort_values("syn_key")

,Unnamed: 0,pre_skid,post_skid,pre_name,post_name,pre_coord,post_coord,roi,pre_root_ids,post_root_ids,pre_side,post_side,type_pre_col,type_post_col,type_pre,type_post,syn_key


In [119]:
new_only = new_df[~new_df["syn_key"].isin(old_keys)].copy()

print(new_only["pre_skid"].value_counts().head(20))
print(new_only["post_skid"].value_counts().head(20))

82590     47
84466     42
84631     42
81527     41
84303     39
75297     38
80994     36
75302     34
82632     33
84500     32
83239     32
83232     31
110806    31
83213     30
83112     30
81541     29
79852     29
34187     29
84354     28
81522     28
Name: pre_skid, dtype: Int64
59381    16
60208    16
34199    14
34205    13
34228    13
35102    11
35471    10
34955     9
42175     9
60458     9
35224     9
99063     9
63772     9
34187     8
57529     8
34931     8
57518     8
35097     7
35304     6
82632     6
Name: post_skid, dtype: Int64


In [127]:
# get skids of neurons to vis
skids = nametable[nametable['name'].str.contains(r'EPG_L3')]
skid_list = skids['skid'].to_list()
skids

,name,root_ids,skid
2,EPG_L3_54939,"[576460752494001689, 576460752504703640]",54939
3,EPG_L3_57518,"[576460752372935125, 576460752440175329, 57646...",57518
4,EPG_L3_57529,"[576460752460921048, 576460752509771516]",57529


In [123]:
# merge

# build keys (make sure coords are already normalized first!)
old_df["syn_key"] = list(zip(old_df["pre_coord"], old_df["post_coord"]))
new_df["syn_key"] = list(zip(new_df["pre_coord"], new_df["post_coord"]))

# build lookup set once
new_keys = set(new_df["syn_key"])

# find rows in old that are missing in new
old_missing = old_df[~old_df["syn_key"].isin(new_keys)].copy()

print(f"rows to add from old: {len(old_missing)}")

# append them to new
merged_df = pd.concat([new_df, old_missing], ignore_index=True)

print(f"final row count: {len(merged_df)}")

rows to add from old: 1271
final row count: 672337


In [129]:
merged_df.to_csv('../syntables/table_csv_files/megalopta_cx_syntable_20-04-26.csv')

In [128]:
'''Synapse plotter'''

'''USER INPUT'''
skid_list = [54939]
roi = r"PB"
view = "normal" # can ignore, just testing...synapses don't work for spleunker for some reason

#######################
if species == "megalopta":
    if roi == 'EB':
        seg_source='graphene://middleauth+https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_EB_v3/'
        print('graphene set for megalopta FB / EB data')
    elif roi == 'PB':
        if view == "normal":
            seg_source='graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a'
            print('graphene set for megalopta PB data')
        elif view == "spleunker":
            seg_source='graphene://middleauth+https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a'
    elif roi == 'PB2':
        seg_source='graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB2'
elif species == "eciton":
    if roi == 'PB':
        seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_PB_v1'
        print('graphene set for eciton PB data')
    elif roi == 'FB':
        seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_FB'
        print('graphene set for eciton FB / EB data')
    elif roi == 'NO':
        seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/eciton_NO_v5'
        print('graphene set for eciton Noduli data')

n = nametable[nametable['skid'].isin(skid_list)]
root_list = (n['root_ids'].explode().tolist())

def to_int_scalar(x):
    # unwrap list/tuple/np.array of length 1
    if isinstance(x, (list, tuple, np.ndarray)):
        if len(x) == 0:
            return None
        x = x[0]
    # unwrap pandas/numpy scalar
    if x is None:
        return None
    return int(x)

root_list = [to_int_scalar(r) for r in root_list]
root_list = [r for r in root_list if r is not None]
print("root_list:", root_list, [type(r) for r in root_list])

# # count the number of segments
count = len(root_list)  
print(f'There are {count} segments')


# Set up viewer options
view_options = {
    "background_color": "white",
    "position": [
       38284.7265625,
       8585.53515625,
       1503.0078125
      ]
}

# Image layer configuration
if roi == "PB2":
    img_src = 'precomputed://https://lweb1569.srv.lu.se/catmaid/data/megalopta/Megalopta_PB_PT2_256'
elif roi == "PB":
    img_src = "precomputed://https://lweb1569.srv.lu.se/catmaid/data/megalopta/Megalopta_PB_PT1_512"
else: 
    img_src = 'precomputed://http://localhost:8000'
#lund_img_source = 'precomputed://https://lweb1569.srv.lu.se/catmaid/data/eciton/Eciton_PB_1026'
img_layer = ImageLayerConfig(name='layer23', source=img_src)

# Segmentation layer configuration with fixed IDs and color mapping
# 'graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_FB'

seg_layer = SegmentationLayerConfig(
    name='seg',
    source=seg_source,
    fixed_ids=root_list  # Use flattened segment IDs - doesn't use df below (in renderstate)
    # fixed_id_colors=color_list  # Color is based on 'color' column from syntable
)


syntable_filt = merged_df[merged_df["roi"].str.contains(roi, na=False)]


syntable_filt = syntable_filt[
    syntable_filt["pre_skid"].isin(skid_list).fillna(False) |
    syntable_filt["post_skid"].isin(skid_list).fillna(False)
]
pre = syntable_filt[syntable_filt["pre_skid"].isin(skid_list).fillna(False)].copy()
post = syntable_filt[syntable_filt["post_skid"].isin(skid_list).fillna(False)].copy()


syntable_filt = pd.concat([pre, post], ignore_index=True)

lines = LineMapper(point_column_a="pre_coord", # use this if you want points connected by lines
                   point_column_b="post_coord")

pre_point = PointMapper(point_column='pre_coord') #to viz pre/post separately, use two anno tabs with point ann
post_point= PointMapper(point_column='post_coord')

anno_layer_lines = AnnotationLayerConfig(
    name='post',
    color='red',
    mapping_rules=lines) 
anno_layer_pre = AnnotationLayerConfig(
    name='pre',
    color='red',
    mapping_rules=pre_point) 
anno_layer_post = AnnotationLayerConfig(
    name='post',
    color='cyan',
    mapping_rules=post_point)


# Build the state with layers and view options
sb = StateBuilder(layers=[img_layer,seg_layer,anno_layer_lines], view_kws=view_options) #seg_layer, img_layer anno_layer

# Render the state as HTML
sb.render_state(syntable_filt, return_as='html', link_text='State')



graphene set for megalopta PB data
root_list: [576460752494001689, 576460752504703640] [<class 'int'>, <class 'int'>]
There are 2 segments


D:\flywire_backup\cave\cave\lib\site-packages\nglui\statebuilder\statebuilder.py:216: UserWarning: Deprecation warning: No target site or url prefix set, using default "seunglab" site. This will switch to "spelunker" in the future.
  warn(


In [22]:
skid_list

[54939]

In [126]:
syntable_filt

,Unnamed: 0,pre_skid,post_skid,pre_name,post_name,pre_coord,post_coord,roi,pre_root_ids,post_root_ids,pre_side,post_side,type_pre_col,type_post_col,type_pre,type_post,syn_key
0,6975777.0,84631,<NA>,Delta7_L3R6_84631,NaN,"(30570.0, 6679.0, 1873.0)","(30573.0, 6678.0, 1875.0)",PB,576460752550433228,576460752501949923,right,NaN,Delta7_L3R6,NaN,Delta7,NaN,"((30570.0, 6679.0, 1873.0), (30573.0, 6678.0, ..."
1,6976048.0,84631,<NA>,Delta7_L3R6_84631,NaN,"(30748.0, 7176.0, 1888.0)","(30748.0, 7201.0, 1888.0)",PB,576460752550433228,576460752339994307,right,NaN,Delta7_L3R6,NaN,Delta7,NaN,"((30748.0, 7176.0, 1888.0), (30748.0, 7201.0, ..."
2,6978054.0,84631,<NA>,Delta7_L3R6_84631,NaN,"(30780.0, 7001.0, 1884.0)","(30784.0, 7021.0, 1884.0)",PB,576460752550433228,576460752312074463,right,NaN,Delta7_L3R6,NaN,Delta7,NaN,"((30780.0, 7001.0, 1884.0), (30784.0, 7021.0, ..."
3,6978240.0,84631,<NA>,Delta7_L3R6_84631,NaN,"(30934.0, 7277.0, 1915.0)","(30917.0, 7294.0, 1914.0)",PB,576460752550433228,576460752342124892,right,NaN,Delta7_L3R6,NaN,Delta7,NaN,"((30934.0, 7277.0, 1915.0), (30917.0, 7294.0, ..."
4,6979030.0,84631,<NA>,Delta7_L3R6_84631,NaN,"(31229.0, 7528.0, 1885.0)","(31212.0, 7546.0, 1886.0)",PB,576460752550433228,576460752377816544,right,NaN,Delta7_L3R6,NaN,Delta7,NaN,"((31229.0, 7528.0, 1885.0), (31212.0, 7546.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10679,7608718.0,80994,84631,Delta7_L6R3_80994,Delta7_L3R6_84631,"(48669.0, 8167.0, 1905.0)","(48694.0, 8174.0, 1904.0)",PB,576460752532127661,576460752550433228,NaN,right,Delta7_L6R3,Delta7_L3R6,Delta7,Delta7,"((48669.0, 8167.0, 1905.0), (48694.0, 8174.0, ..."
10680,7608726.0,<NA>,84631,NaN,Delta7_L3R6_84631,"(48765.0, 8201.0, 1906.0)","(48761.0, 8218.0, 1906.0)",PB,576460752311419021,576460752550433228,NaN,right,NaN,Delta7_L3R6,NaN,Delta7,"((48765.0, 8201.0, 1906.0), (48761.0, 8218.0, ..."
10681,7613019.0,60458,84631,EPG_L6_60458,Delta7_L3R6_84631,"(48523.0, 9238.0, 1916.0)","(48544.0, 9218.0, 1915.0)",PB,576460752541325943,576460752550433228,left,right,EPG_L6,Delta7_L3R6,EPG,Delta7,"((48523.0, 9238.0, 1916.0), (48544.0, 9218.0, ..."
10682,7637206.0,<NA>,84631,NaN,Delta7_L3R6_84631,"(45808.0, 6967.0, 1968.0)","(45823.0, 6948.0, 1967.0)",PB,576460752361665949,576460752550433228,NaN,right,NaN,Delta7_L3R6,NaN,Delta7,"((45808.0, 6967.0, 1968.0), (45823.0, 6948.0, ..."


In [20]:
len(syntable[syntable["pre_skid"].isin(skid_list).fillna(False)])

7052

In [21]:
len(syntable[syntable["post_skid"].isin(skid_list).fillna(False)])

5757

## neuroglancer-catmaid api

In [5]:
import requests

In [6]:
d7 = nametable[nametable["name"].str.contains(r"L1L9", na=False)]
d = d7['skid'].to_list()
d

[81004, 84275, 81014, 76634, 82977, 78122]

In [ ]:
# megalopta PB1 stack = 41

In [15]:
jq = {
  "skeleton_ids": [
    81004, 84275
  ],
  "project_id": 8,
  "stack_id": 42,
  "clear_cache": False
}

In [16]:
print(json.dumps(jq, indent=2))

{
  "skeleton_ids": [
    81004,
    84275
  ],
  "project_id": 8,
  "stack_id": 42,
  "clear_cache": false
}


In [19]:
r = requests.post(
    "http://localhost:8000/links/",
    json=jq
)

print(r.json())
# print(r.status_code)
# print(r.text)

[{'megalopta_pb2_datastack': {'viewer_state': 'https://spelunker.cave-explorer.org/#!%7B%22concurrentDownloads%22:32,%22position%22:%5B50342.0,4514.0,2342.5%5D,%22layout%22:%22xy-3d%22,%22dimensions%22:%7B%22x%22:%5B1e-08,%22m%22%5D,%22y%22:%5B1e-08,%22m%22%5D,%22z%22:%5B5e-08,%22m%22%5D%7D,%22layers%22:%5B%7B%22type%22:%22image%22,%22source%22:%5B%7B%22url%22:%22precomputed://https://lweb1569.srv.lu.se/catmaid/data/megalopta/Megalopta_PB_PT2_256%22,%22transform%22:%7B%22outputDimensions%22:%7B%22x%22:%5B1e-08,%22m%22%5D,%22y%22:%5B1e-08,%22m%22%5D,%22z%22:%5B5e-08,%22m%22%5D%7D%7D,%22subsources%22:%7B%7D,%22enableDefaultSubsources%22:true%7D%5D,%22name%22:%22imagery%22%7D,%7B%22type%22:%22segmentation%22,%22source%22:%5B%7B%22url%22:%22graphene://middleauth+https://local.cave.braininbrain.org/segmentation/table/megalopta_PB2%22,%22transform%22:%7B%22outputDimensions%22:%7B%22x%22:%5B1e-08,%22m%22%5D,%22y%22:%5B1e-08,%22m%22%5D,%22z%22:%5B5e-08,%22m%22%5D%7D%7D,%22subsources%22:%7B%7D,

In [43]:
r = requests.post(
    "http://localhost:8000/links/",
    json=jq
)

print(r.status_code)
print(r.text)

500
Internal Server Error


In [126]:
syntable[syntable["pre_name"].str.contains(r"Delta7_L2R7", na=False)]

,pre_coord,post_coord,roi,pre_root_id,post_root_id,type_pre_col,type_post_col,type_pre,type_post,pre_name,pre_skid,post_name,post_skid
2117706,"(10417, 27236, 1576)","(10449, 27210, 1577)",PB,576460752475443648,576460752384933962,NaN,NaN,NaN,NaN,Delta7_L2R7_FS,126978.0,NaN,NaN
2121253,"(9945, 27884, 1450)","(9915, 27884, 1450)",PB,576460752615871276,576460752441059456,NaN,NaN,NaN,NaN,Delta7_L2R7_FS,139243.0,NaN,NaN
2121254,"(9941, 27867, 1451)","(9968, 27867, 1451)",PB,576460752615871276,576460752441062528,NaN,NaN,NaN,NaN,Delta7_L2R7_FS,139243.0,NaN,NaN
2121466,"(9881, 27910, 1599)","(9881, 27932, 1600)",PB,576460752615871276,576460752407459265,NaN,NaN,NaN,NaN,Delta7_L2R7_FS,139243.0,NaN,NaN
2123659,"(17639, 27765, 1559)","(17663, 27740, 1559)",PB,576460752480161453,576460752423351153,NaN,NaN,NaN,NaN,Delta7_L2R7_FS,128800.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2974481,"(3114, 32380, 2438)","(3137, 32356, 2437)",PB,576460752475296830,576460752427921090,NaN,NaN,NaN,NaN,Delta7_L2R7_FS,128707.0,NaN,NaN
2974483,"(3246, 32480, 2438)","(3209, 32476, 2438)",PB,576460752470968168,576460752335482834,NaN,NaN,NaN,NaN,Delta7_L2R7_FS,128697.0,NaN,NaN
2974484,"(3245, 32464, 2439)","(3239, 32428, 2439)",PB,576460752470968168,576460752335458770,NaN,NaN,NaN,NaN,Delta7_L2R7_FS,128697.0,NaN,NaN
2974486,"(3128, 32373, 2441)","(3158, 32358, 2440)",PB,576460752475296830,576460752334873554,NaN,NaN,NaN,NaN,Delta7_L2R7_FS,128707.0,NaN,NaN


In [83]:
dup_mask = syntable.duplicated(subset=['pre_coord', 'post_coord'], keep=False)
dups = syntable[dup_mask]


# electrotonic distance

In [ ]:
syntable

In [ ]:
'''Synapse plotter'''

'''USER INPUT'''
use_flywire_only = False # set to true to use only flywire segment ids (if using Valentin's .csv file)


skid = 55952


###########################################

skid_list = [skid] # if false, add skid(s) list here

root_list = nametable['root_ids'].loc[nametable['skid']==skid].iloc[0]

# # count the number of segments
count = len(root_list)  
print(f'There are {count} segments')

lal = syntable[syntable['partner_name'].str.contains('LAL')]
mbuv = syntable[syntable['partner_name'].str.contains('MBUv')]
mbud = syntable[syntable['partner_name'].str.contains('MBUd')]
lbu = syntable[syntable['partner_name'].str.contains('LBU')]
    
#pre_syn = syntable[syntable['pre_skid'].isin(skid_list)]
#pre_syn = pre_syn['pre_coord']

#post_syn = syntable[syntable['post_skid'].isin(skid_list)]
#post_syn = post_syn['post_coord']

# Set up viewer options
view_options = {
    'background_color': 'white'
}

# Image layer configuration
img_source = 'precomputed://http://localhost:8000'
img_layer = ImageLayerConfig(name='layer23', source=img_source)

# Segmentation layer configuration with fixed IDs and color mapping
# seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_syntable_v3'
seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_EB_v3'
seg_layer = SegmentationLayerConfig(
    name='seg',
    source=seg_source,
    fixed_ids=root_list  # Use flattened segment IDs - doesn't use df below (in renderstate)
    # fixed_id_colors=color_list  # Color is based on 'color' column from syntable
)

# Line mapper for pre and post voxel coordinates
lines = LineMapper(point_column_a='pre_coord', 
                   point_column_b='post_coord')

pointA = PointMapper(point_column='pre_coord')

pointB= PointMapper(point_column='post_coord')

point_list = [pointA]

'''to viz pre/post separately, use two anno tabs with point ann'''

# # Annotation layer configuration
anno_layer = AnnotationLayerConfig(
    name='annos',
    mapping_rules=pointB) # lines

# Build the state with layers and view options
sb = StateBuilder(layers=[img_layer,seg_layer,anno_layer], view_kws=view_options) #seg_layer, img_layer anno_layer

# Render the state as HTML
sb.render_state(lbu, return_as='html', link_text='State')
# sb.render_state()


# adding colors

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcl
import seaborn as sns

# colors = {
#     'delta7': "#47B03F", 
#     'EPG': "#009A90",  
#     'PEG': "#DF7878",  
#     'PEN1': "#F09E00",  
#     'PEN2': "#6EBADD", 
#     'LBU': "#45CE68", 
#     'MBUd': "#F0CD00", 
#     'MBUv': "#B878A9",  
#     'ExR': "#EF506C",  
#     'LAL': "#92C77D", 
#     'LNOm': "#DF4C21",
#     'LNOs': "#387DBE"    
# }


colors = {
    'L2': "#6bff9c", 
    'L3': "#56ECE2",  
    'L4': "#FF4437",  
    'L5': "#F09E00",  
    'L6': "#56ECE2", 
    'L7': "#45CE68", 
    'L8': "#FFDA00", 
    'L9': "#E95DFC"  
}

# Extract the list of colors for Seaborn palette
palette_list = list(colors.values())

# Create a Seaborn color palette
pal = sns.color_palette(palette_list)

# Set the palette globally for Seaborn plots
sns.set_palette(pal)

pal


In [ ]:
nametable

In [ ]:
'''to color based on column'''
#############################################
def color_matching(name_value):
    # Ensure name_value is a string before attempting substring matching
    if isinstance(name_value, str):
        for k, v in colors.items():
            if k in name_value:
                return v
    # Default color if no match or if name_value is not a string
    return '#000000'


nametable['color'] = nametable['name'].apply(color_matching)

# syntable['color'] = syntable.apply(
#     lambda row: color_matching(row['pre_neuron_name']) or color_matching(row['post_neuron_name']), 
#     axis=1
# )

In [ ]:
'''to color cyclically'''
# neurons colored for morphology images
color_values = list(colors.values())

# Repeat the color values to match the length of nametable
repeated_colors = color_values * (len(nametable) // len(color_values)) + color_values[:len(nametable) % len(color_values)]

# Add the repeated colors as a new column in the DataFrame
nametable['color'] = repeated_colors

nametable




In [ ]:
ntable = n_morphology.head(5)
# Convert 'root_ids' to integers using Python's built-in int type
ntable['root_ids'] = ntable['root_ids'].apply(lambda x: int(x) if pd.notnull(x) else None)

In [ ]:
'''Synapse plotter'''

'''USER INPUT'''
use_flywire_only = False # set to true to use only flywire segment ids (if using Valentin's .csv file)


skid = 81004


###########################################

skid_list = [skid] # if false, add skid(s) list here

root_list = nametable['root_ids'].loc[nametable['skid']==skid].iloc[0]

# # count the number of segments
count = len(root_list)  
print(f'There are {count} segments')

if use_flywire_only == True:
    # filter by root_id
    syntable_filt = syntable[(syntable['pre_root_ids'].isin(root_list)) | (syntable['post_root_ids'].isin(root_list))]
else: 
    # filter using catmaid skid
    # this will miss branches that have not been merged, but will add branches not yet found in flywire
    # may also introduce some error if skeleton has incorrect branch in catmaid...
    #syntable_filt = syntable[(syntable['pre_skid'].isin(skid_list)) | (syntable['post_skid'].isin(skid_list))]
    
    pre_syn = syntable[syntable['pre_skid'].isin(skid_list)]
    #pre_syn = pre_syn['pre_coord']
    
    post_syn = syntable[syntable['post_skid'].isin(skid_list)]
    #post_syn = post_syn['post_coord']

# Set up viewer options
view_options = {
    'background_color': 'white'
}

# Image layer configuration
img_source = 'precomputed://http://localhost:8000'
img_layer = ImageLayerConfig(name='layer23', source=img_source)

# Segmentation layer configuration with fixed IDs and color mapping
# seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_syntable_v3'
seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a'
seg_layer = SegmentationLayerConfig(
    source=seg_source,
    name='seg',
    selected_ids_column='root_ids',
    fixed_ids=root_list,  # Use flattened segment IDs - doesn't use df below (in renderstate)
    color_column='color'
)

# Line mapper for pre and post voxel coordinates
lines = LineMapper(point_column_a='pre_coord', 
                   point_column_b='post_coord')

pointA = PointMapper(point_column='pre_coord')

pointB= PointMapper(point_column='post_coord')

point_list = [pointA]

'''to viz pre/post separately, use two anno tabs with point ann'''

# # Annotation layer configuration
anno_layer = AnnotationLayerConfig(
    name='annos',
    mapping_rules=pointA) # lines

# Build the state with layers and view options
sb = StateBuilder(layers=[img_layer,seg_layer], view_kws=view_options) #seg_layer, img_layer anno_layer

# Render the state as HTML
sb.render_state(ntable, return_as='html', link_text='State')
# sb.render_state()


### 

# JSON generator

In [ ]:
jtable['color']

In [ ]:

#jtable = nametable[nametable['name'].str.contains('PEN')]

'''for pens'''

colors = {
    'L2': "#8000ff", 
    'L3': "#1996f3",  
    'L4': "#4df3ce",  
    'L5': "#F09E00",  
    'L6': "#56ECE2", 
    'L7': "#45CE68", 
    'L8': "#FFDA00", 
    'L9': "#E95DFC"  
}

colors = {
    "Color 1": "#8000ff",
    "Color 2": "#1996f3",
    "Color 3": "#4df3ce",
    "Color 4": "#b2f396",
    "Color 5": "#ff964f",
    "Color 6": "#ff0000"
}

'''to color cyclically'''
# neurons colored for morphology images
color_values = list(colors.values())

# Repeat the color values to match the length of nametable
repeated_colors = color_values * (len(jtable) // len(color_values)) + color_values[:len(jtable) % len(color_values)]

# Add the repeated colors as a new column in the DataFrame
jtable['color'] = repeated_colors

jtable

# 1. Split 'root_ids' on commas to create list-of-strings
jtable['root_ids'] = jtable['root_ids'].str.split(',')

# 2. Explode to expand rows
jtable = jtable.explode('root_ids', ignore_index=True)

# 3. (Optional) convert each root_id to a string or integer
jtable['root_ids'] = jtable['root_ids'].astype(str)

jtable

In [ ]:
jtable

In [ ]:
def find_layer_label(neuron_name, possible_labels):
    """
    Return the first label in `possible_labels` that is found as a substring in `neuron_name`.
    Returns None if no label is found.
    """
    for label in possible_labels:
        if label in neuron_name:
            return label
    return None

# Assume `jtable` is your DataFrame

layer_keys = list(colors.keys())  # e.g. ["L2","L3","L4","L5","L6","L7","L8","L9"]

jtable['layer_label'] = jtable['name'].apply(lambda x: find_layer_label(x, layer_keys))

filtered_df = jtable[jtable['layer_label'].notnull()].copy()

grouped = filtered_df.groupby('layer_label')

def split_root_ids(root_ids_value):
    """
    Takes a single 'root_ids' field (which might be a string of comma-separated values)
    and returns a list of stripped strings.
    """
    # In case the row is already a string of IDs, separated by commas
    if isinstance(root_ids_value, str):
        return [x.strip() for x in root_ids_value.split(',')]
    
    # If it's just a single ID (int or something else), make it a list
    # (though best practice is to ensure your DataFrame column is consistent)
    return [str(root_ids_value)]


label_to_data = {}

for label, subdf in grouped:
    # We will accumulate all 'segments' and 'segmentColors' for the *entire group* here
    segments_list = []
    segment_colors_dict = {}

    for _, row in subdf.iterrows():
        # Expand root_ids to a list (handling multi-ID rows)
        root_ids_for_this_row = split_root_ids(row['root_ids'])

        # For each ID in that row, add to segments_list
        for seg_id in root_ids_for_this_row:
            seg_id_str = str(seg_id)  # ensure it's a string
            segments_list.append(seg_id_str)

            # The color for *this neuron* is row['color']
            # so we map this exact root ID to that color
            segment_colors_dict[seg_id_str] = row['color']

    # Now we have all the segments and their colors for this label
    label_to_data[label] = {
        "segments": segments_list,
        "segmentColors": segment_colors_dict
    }

layer_template = {
  "source": "graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a",
  "type": "segmentation_with_graph",
  "selectedAlpha": 0.3,
  "segmentColors": {},  # fill in
  "segments": [],       # fill in
  "skeletonRendering": {
    "mode2d": "lines_and_points",
    "mode3d": "lines"
  },
  "graphOperationMarker": [
    { "annotations": [], "tags": [] },
    { "annotations": [], "tags": [] }
  ],
  "pathFinder": {
    "color": "#ffff00",
    "pathObject": {
      "annotationPath": { "annotations": [], "tags": [] },
      "hasPath": False
    }
  },
  "name": ""  # fill in (e.g. "L2", "L3", etc.)
}

def build_ng_layer(layer_name, data_dict):
    """
    Given a label (e.g., 'L2') and a dict with 'segments' + 'segmentColors',
    build the Neuroglancer layer JSON object.
    """
    return {
        "source": "graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a",
        "type": "segmentation_with_graph",
        "selectedAlpha": 0.3,
        "segmentColors": data_dict["segmentColors"],
        "segments": data_dict["segments"],
        "skeletonRendering": {
            "mode2d": "lines_and_points",
            "mode3d": "lines"
        },
        "graphOperationMarker": [
            { "annotations": [], "tags": [] },
            { "annotations": [], "tags": [] }
        ],
        "pathFinder": {
            "color": "#ffff00",
            "pathObject": {
                "annotationPath": {
                    "annotations": [],
                    "tags": []
                },
                "hasPath": False
            }
        },
        "name": layer_name
    }

new_layers = []
for label, data_dict in label_to_data.items():
    ng_layer = build_ng_layer(label, data_dict)
    new_layers.append(ng_layer)

import json

# 5.1: Read the existing viewer state
with open("viewer_state.json", "r", encoding="utf-8") as f:
    viewer_data = json.load(f)

# 5.2: Insert our new layers
if "layers" not in viewer_data:
    viewer_data["layers"] = []

viewer_data["layers"].extend(new_layers)

# 5.3: Write the updated JSON out to a new file (or overwrite)
with open("viewer_state_updated.json", "w", encoding="utf-8") as f:
    json.dump(viewer_data, f, indent=2)



In [ ]:
viewer_data

# delta7 presynaptic site plotter

In [ ]:
'''Synapse plotter'''

'''USER INPUT'''
use_flywire_only = False # set to true to use only flywire segment ids (if using Valentin's .csv file)


skid = 82595


###########################################

skid_list = [skid] # if false, add skid(s) list here

root_list = nametable['root_ids'].loc[nametable['skid']==skid].iloc[0]

# # count the number of segments
count = len(root_list)  
print(f'There are {count} segments')

if use_flywire_only == True:
    # filter by root_id
    syntable_filt = syntable[(syntable['pre_root_ids'].isin(root_list)) | (syntable['post_root_ids'].isin(root_list))]
else: 
    # filter using catmaid skid
    # this will miss branches that have not been merged, but will add branches not yet found in flywire
    # may also introduce some error if skeleton has incorrect branch in catmaid...
    #syntable_filt = syntable[(syntable['pre_skid'].isin(skid_list)) | (syntable['post_skid'].isin(skid_list))]
    
    pre_syn = syntable[syntable['pre_skid'].isin(skid_list)]
    #pre_syn = pre_syn['pre_coord']
    
    post_syn = syntable[syntable['post_skid'].isin(skid_list)]
    #post_syn = post_syn['post_coord']

# Set up viewer options
view_options = {
    'background_color': 'white'
}

# Image layer configuration
img_source = 'precomputed://http://localhost:8000'
img_layer = ImageLayerConfig(name='layer23', source=img_source)

# Segmentation layer configuration with fixed IDs and color mapping
# seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_syntable_v3'
seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a'
seg_layer = SegmentationLayerConfig(
    name='seg',
    source=seg_source,
    fixed_ids=root_list  # Use flattened segment IDs
    # fixed_id_colors=color_list  # Color is based on 'color' column from syntable
)

# Line mapper for pre and post voxel coordinates
lines = LineMapper(point_column_a='pre_coord', 
                   point_column_b='post_coord')

pointA = PointMapper(point_column='pre_coord')

pointB= PointMapper(point_column='post_coord')

point_list = [pointA]

'''to viz pre/post separately, use two anno tabs with point ann'''

# # Annotation layer configuration
anno_layer = AnnotationLayerConfig(
    name='annos',
    mapping_rules=pointB) # lines

# Build the state with layers and view options
sb = Statsyntableuilder(layers=[img_layer,seg_layer,anno_layer], view_kws=view_options) #seg_layer, img_layer anno_layer

# Render the state as HTML
sb.render_state(post_syn, return_as='html', link_text='State')
# sb.render_state()


----------------------------------------------------------------------------
### for colored segments



In [ ]:
# Example value for x
x = len(flat_root_idss)

# Extract the color value from the array
n_color = filt_skid['color'].values

# Repeat the color value based on x
n_color_list = [n_color[0]] * x  # Assuming n_color has at least one element
print(repeated_colors)

In [ ]:
# Convert each cell in 'root_ids' from a comma-separated string to a list of integers
# Convert each cell in 'root_ids' from a comma-separated string to a list of integers, ignoring empty strings
nametable['root_ids'] = nametable['root_ids'].apply(lambda x: [int(seg) for seg in x.split(',') if seg.strip()])

nametable['root_ids']

In [ ]:
# Define the color palette dictionary
n_colors = {'EPG':'#5f57db', 'PEG':'#6aa5c7', 'PEN':'#d76d4c', 'delta7':'#6aae5c'}
syn_colors = {'EPG':'#5f57db', 'PEG':'#6aa5c7', 'PEN':'#d76d4c', 'delta7':'#6aae5c'}

# Initialize a new 'color' column to None or any default value
PB['n_color'] = None

# Use apply to fill the color column based on the 'type' column
nametable['color'] = nametable['type'].apply(lambda x: n_colors.get(x, None)) # this won't work until root_ids are added to PB_filt
# PB['syn_colors'] = PB['type'].apply(lambda x: n_colors.get(x, None))

# Example value for x
x = len(flat_root_idss)

# Extract the color value from the array
n_color = filt_skid['color'].values

# Repeat the color value based on x
n_color_list = [n_color[0]] * x  # Assuming n_color has at least one element
print(repeated_colors)

In [ ]:
seg_layer = SegmentationLayerConfig(seg_source, selected_ids_column='post_pt_root_id')

postsyn_mapper = LineMapper(
    point_column_a='pre_pt_position',
    point_column_b='ctr_pt_position',
    mapping_set='post',     # This tells the LineMapper to use the dataframe passed with the dictionary key `post`
)
postsyn_annos = AnnotationLayerConfig('post', color='#00CCCC', mapping_rules=postsyn_mapper)

presyn_mapper = LineMapper(
    point_column_a='ctr_pt_position',
    point_column_b='post_pt_position',
    mapping_set='pre',      # This tells the LineMapper to use the dataframe passed with the dictionary key `pre`
)
presyn_annos = AnnotationLayerConfig('pre', color='#CC1111', mapping_rules=presyn_mapper)

sb = Statsyntableuilder([seg_layer, postsyn_annos, presyn_annos], client=client)

# Note that the data is passed as a dictionary with the same keys as the mapping rules above.
sb.render_state(
    {
        'post': post_syn_df,
        'pre': pre_syn_df,
    },
    return_as='html'
)

In [ ]:
# Synapse plotter
skid_list = [64077]

###########################################################################

# Filter the skid
filt_skid = nametable[nametable['skid'].isin(skid_list)]

# Define a function to split and convert segment IDs
def convert_segment_ids(value):
    # Split by comma, then convert each part to an integer
    return [int(v) for v in value.split(',')]
    
###########
# color_list = []

# for index, row in filt_skid.iterrows():
#     split_ids = row['root_ids'].split(',')
#     if len(split_ids) > 1:
#         repeats = len(split_ids)
#         color = row['color']
#         n_color_list = [color] * repeats
#         color_list.extend(n_color_list)
#     elif len(split_ids) == 1:
#         color_list.append(row['color'])

###########
        
# Apply the function to the 'Latest segment IDs' column
filt_skid['root_ids'] = filt_skid['root_ids'].apply(convert_segment_ids)

# Now root_idss will be a list of lists containing integers
root_idss = filt_skid['root_ids'].values

# Count the number of segments
count = sum(len(root_ids) for root_ids in root_idss)  # sum all segments from the lists
print(f'There are {count} segments')

# Flatten the root_idss into a single list of integers
flat_root_idss = [root_ids for sublist in root_idss for root_ids in sublist]



# Filter PB dataframe to get rows matching pre_skid or post_skid
PB_filt = PB[(PB['pre_skid'].isin(skid_list)) & (PB['post_skid'].isin(skid_list))]

# Set up viewer options
view_options = {
    'background_color': 'white'
}

# Image layer configuration
img_source = 'precomputed://http://localhost:8000'
img_layer = ImageLayerConfig(name='layer23', source=img_source)

# Segmentation layer configuration with fixed IDs and color mapping
seg_source = 'graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a'
seg_layer = SegmentationLayerConfig(
    name='seg',
    source=seg_source,
    fixed_ids=flat_root_idss  # Use flattened segment IDs
    #fixed_id_colors=color_list  # Color is based on 'color' column from PB
)

# Line mapper for pre and post voxel coordinates
lines = LineMapper(point_column_a='pre_coord', 
                   point_column_b='post_coord')

# Annotation layer configuration
anno_layer = AnnotationLayerConfig(
    name='annos',
    mapping_rules=lines)

# Build the state with layers and view options
sb = Statsyntableuilder(layers=[img_layer, seg_layer, anno_layer], view_kws=view_options) #anno_layer

# Render the state as HTML
sb.render_state(PB_filt, return_as='html', link_text='State')  # Or sb.render_state(return_as='json') if you need JSON output


# 

# 